# Cross-Method and Temporal-Order Stress Validation

This notebook reproduces the **secondary cross-model validation and temporal-order stress test** reported in the manuscript.

It evaluates four fixed model families — **Random Forest (RF), RBF-SVM, XGBoost, and LSTM** — under:

- **generalized subject-disjoint evaluation** (5 outer folds);
- **individualized subject-task evaluation** (10 repeated stratified splits per eligible pair);
- **original versus temporally shuffled trajectories**.

The same deterministic within-trial frame permutation is reused across all models and regimes. Individualized results are first averaged across repeated splits at the **subject-task pair** level. Temporal-shuffle sensitivity is then additionally aggregated at the **subject** level and tested using an exact two-sided sign-flip test with Holm correction across the four model families within each dataset.

### Data used

- **REHAB24-6:** this notebook deliberately loads the verified `1072 × 100 × 10` joint-angle tensor exported by the primary REHAB24-6 notebook. This avoids reconstructing the dataset through the earlier ambiguous raw-file resolver and guarantees that the secondary experiment uses the same representation as the primary analysis.
- **IntelliRehabDS:** the common `100 × 10` joint-angle representation is rebuilt from the public `Simplified` skeleton files (or loaded from the validated cache).

The notebook is intended to generate the manuscript's cross-method performance table and subject-aware temporal-shuffle sensitivity table from one reproducible workflow.


## Expected Google Drive layout

Edit the paths in the configuration cell if your local layout differs.

```text
MyDrive/
├── MQM_cross_method_validation/
│   ├── inputs/
│   │   └── rehab24_6_primary_angles_1072.npz
│   ├── processed/
│   ├── manifests/
│   └── outputs/
└── NeuroYOLO_IRDS (1)/
    └── SkeletonData/
        └── SkeletonData/
            └── Simplified/
```

`rehab24_6_primary_angles_1072.npz` is generated by the primary REHAB24-6 analysis notebook and must contain:
`X`, `y`, `subject_id`, `task_id`, and `sample_id`.


In [ ]:
# Colab: mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab; configure local paths in the next cell.")


In [ ]:
# Imports and deterministic setup
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_CUDNN_DETERMINISTIC", "1")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("Install xgboost before running this notebook: !pip -q install xgboost") from exc

try:
    import tensorflow as tf
    from tensorflow import keras
except ImportError as exc:
    raise ImportError("Install TensorFlow before running this notebook.") from exc

BASE_SEED = 42

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def stable_seed(*parts: Any, modulo: int = 2_147_483_647) -> int:
    """Session-independent seed derived from arbitrary identifiers."""
    payload = "||".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return int.from_bytes(digest[:8], "little") % modulo

@dataclass
class DatasetBundle:
    name: str
    X: np.ndarray
    y: np.ndarray
    subject_id: np.ndarray
    task_id: np.ndarray
    sample_id: np.ndarray
    fingerprint: str

set_global_seed(BASE_SEED)
print("TensorFlow:", tf.__version__)
print("Base seed:", BASE_SEED)


In [ ]:
# Paths and protocol configuration
ROOT = Path("/content/drive/MyDrive/MQM_cross_method_validation")
INPUT_DIR = ROOT / "inputs"
PROCESSED_DIR = ROOT / "processed"
MANIFEST_DIR = ROOT / "manifests"
OUTPUT_DIR = ROOT / "outputs"

for directory in (INPUT_DIR, PROCESSED_DIR, MANIFEST_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Exact REHAB24-6 representation exported by the primary analysis notebook.
PRIMARY_REHAB_EXPORT = INPUT_DIR / "rehab24_6_primary_angles_1072.npz"

# IntelliRehabDS public Simplified skeleton files.
INTELLI_BASE_DIR = Path(
    "/content/drive/MyDrive/NeuroYOLO_IRDS (1)/SkeletonData/SkeletonData"
)
INTELLI_SIMPLIFIED_DIR = INTELLI_BASE_DIR / "Simplified"
INTELLI_CACHE = PROCESSED_DIR / "intellirehabds_common_angles_v2.npz"

MANIFEST_FILES = {
    "REHAB24-6": MANIFEST_DIR / "rehab24_6_primary_tensor_cross_method_splits_v2.json",
    "IntelliRehabDS": MANIFEST_DIR / "intellirehabds_cross_method_splits_v2.json",
}

REBUILD_INTELLI_PROCESSED_DATA = False
REBUILD_MANIFESTS = False

TARGET_LEN = 100
REPRESENTATION_VERSION = "coco17_common_10_angles_v2"

N_GENERALIZED_FOLDS = 5
N_INDIVIDUALIZED_REPEATS = 10
BASE_TEST_FRACTION = 0.30
MIN_CLASS_COUNT = 2
MIN_TOTAL_COUNT = 6

SHUFFLE_SEED = 20260803

EXPECTED_INDIVIDUALIZED_PAIRS = {
    "REHAB24-6": 46,
    "IntelliRehabDS": 16,
}
STRICT_PAIR_COUNT_CHECK = True

# Fingerprints/sanity values from the final manuscript analysis.
EXPECTED_REHAB_FINGERPRINT_PREFIX = "77af2936b65196ec"
EXPECTED_INTELLI_FINGERPRINT = (
    "98d613532c7f556d5816f0dab7b82dcb40510c206622239c227f16749589fd58"
)

MODELS_TO_RUN = ["RF", "SVM", "XGBoost", "LSTM"]
CONDITIONS_TO_RUN = ["original", "shuffled"]
REGIMES_TO_RUN = ["generalized", "individualized"]

@dataclass(frozen=True)
class RFConfig:
    n_estimators: int = 300
    max_depth: int | None = None
    min_samples_leaf: int = 1
    class_weight: str = "balanced"
    n_jobs: int = -1

@dataclass(frozen=True)
class SVMConfig:
    C: float = 1.0
    gamma: str = "scale"
    class_weight: str = "balanced"

@dataclass(frozen=True)
class XGBConfig:
    n_estimators: int = 300
    max_depth: int = 5
    learning_rate: float = 0.05
    subsample: float = 0.80
    colsample_bytree: float = 0.80
    reg_lambda: float = 1.0
    min_child_weight: float = 1.0
    n_jobs: int = -1

@dataclass(frozen=True)
class LSTMConfig:
    units: int = 64
    dropout: float = 0.30
    learning_rate: float = 1e-3
    epochs: int = 40
    batch_size: int = 32

RF_CONFIG = RFConfig()
SVM_CONFIG = SVMConfig()
XGB_CONFIG = XGBConfig()
LSTM_CONFIG = LSTMConfig()

CONFIG_PAYLOAD = {
    "representation_version": REPRESENTATION_VERSION,
    "base_seed": BASE_SEED,
    "shuffle_seed": SHUFFLE_SEED,
    "n_generalized_folds": N_GENERALIZED_FOLDS,
    "n_individualized_repeats": N_INDIVIDUALIZED_REPEATS,
    "base_test_fraction": BASE_TEST_FRACTION,
    "min_class_count": MIN_CLASS_COUNT,
    "min_total_count": MIN_TOTAL_COUNT,
    "rf": asdict(RF_CONFIG),
    "svm": asdict(SVM_CONFIG),
    "xgboost": asdict(XGB_CONFIG),
    "lstm": asdict(LSTM_CONFIG),
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(CONFIG_PAYLOAD, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

print(json.dumps(CONFIG_PAYLOAD, indent=2))
print("Configuration hash:", CONFIG_HASH)


## Common joint-angle representation

IntelliRehabDS is mapped to the same COCO-like joint topology and converted to the common 10-angle representation used in the manuscript. REHAB24-6 is loaded directly from the verified export produced by the primary notebook.


In [ ]:
# Shared representation functions
ANGLE_NAMES = [
    "L_elbow", "R_elbow",
    "L_knee", "R_knee",
    "L_shoulder", "R_shoulder",
    "L_hip", "R_hip",
    "torso_L_dev", "torso_R_dev",
]

def normalize_skeleton_length(sequence: np.ndarray, target_len: int = 100) -> np.ndarray:
    """Linear interpolation along time for a sequence shaped (T, J, D)."""
    sequence = np.asarray(sequence, dtype=np.float32)
    if sequence.ndim != 3:
        raise ValueError(f"Expected skeleton shape (T,J,D), received {sequence.shape}.")
    old_len = sequence.shape[0]
    if old_len < 2:
        raise ValueError("A repetition must contain at least two frames.")
    if old_len == target_len:
        return sequence.copy()

    new_position = np.linspace(0, old_len - 1, target_len)
    lower = np.floor(new_position).astype(int)
    upper = np.clip(lower + 1, 0, old_len - 1)
    weight = (new_position - lower).astype(np.float32)
    return (
        (1.0 - weight)[:, None, None] * sequence[lower]
        + weight[:, None, None] * sequence[upper]
    ).astype(np.float32)

def angle_abc(A: np.ndarray, B: np.ndarray, C: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Angle at B for arrays shaped (...,D)."""
    BA = A - B
    BC = C - B
    numerator = np.sum(BA * BC, axis=-1)
    denominator = np.linalg.norm(BA, axis=-1) * np.linalg.norm(BC, axis=-1)
    cosine = numerator / (denominator + eps)
    return np.arccos(np.clip(cosine, -1.0, 1.0))

def unit_vector(start: np.ndarray, end: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    vector = end - start
    norm = np.linalg.norm(vector, axis=-1, keepdims=True)
    return vector / (norm + eps)

def compute_common_angle_timeseries(coco: np.ndarray) -> np.ndarray:
    """
    Compute the manuscript's common 10-angle representation.

    Input:
        coco: (100,17,D), D in {2,3}
    Output:
        angles: (100,10), radians
    """
    coco = np.asarray(coco, dtype=np.float32)
    if coco.ndim != 3 or coco.shape[1] != 17:
        raise ValueError(f"Expected (T,17,D), received {coco.shape}.")

    LS, RS = coco[:, 5], coco[:, 6]
    LE, RE = coco[:, 7], coco[:, 8]
    LW, RW = coco[:, 9], coco[:, 10]
    LH, RH = coco[:, 11], coco[:, 12]
    LK, RK = coco[:, 13], coco[:, 14]
    LA, RA = coco[:, 15], coco[:, 16]

    left_torso = unit_vector(LH, LS)
    right_torso = unit_vector(RH, RS)
    mid_torso = unit_vector((LH + RH) / 2.0, (LS + RS) / 2.0)

    angles = np.stack(
        [
            angle_abc(LS, LE, LW),
            angle_abc(RS, RE, RW),
            angle_abc(LH, LK, LA),
            angle_abc(RH, RK, RA),
            angle_abc(LE, LS, LH),
            angle_abc(RE, RS, RH),
            angle_abc(LS, LH, LK),
            angle_abc(RS, RH, RK),
            np.arccos(np.clip(np.sum(left_torso * mid_torso, axis=-1), -1.0, 1.0)),
            np.arccos(np.clip(np.sum(right_torso * mid_torso, axis=-1), -1.0, 1.0)),
        ],
        axis=1,
    ).astype(np.float32)

    if angles.shape != (TARGET_LEN, 10):
        raise ValueError(f"Unexpected angle shape: {angles.shape}")
    if not np.isfinite(angles).all():
        raise ValueError("Angle sequence contains non-finite values.")
    return angles


In [ ]:
# IntelliRehabDS raw loader adapted from the supplied primary notebook
INTELLI_TO_COCO = {
    3: 0,
    4: 5,
    8: 6,
    5: 7,
    9: 8,
    6: 9,
    10: 10,
    12: 11,
    16: 12,
    13: 13,
    17: 14,
    14: 15,
    18: 16,
}

def parse_intelli_filename(path: Path) -> dict[str, Any]:
    """
    Expected form:
        subject_session_gesture_repetition_correctness_posture.txt
    """
    parts = path.stem.split("_")
    if len(parts) < 6:
        raise ValueError(f"Unexpected IntelliRehabDS filename: {path.name}")
    return {
        "subject": int(parts[0]),
        "session": int(parts[1]),
        "gesture": int(parts[2]),
        "repetition": int(parts[3]),
        "correctness": int(parts[4]),
        "posture": "_".join(parts[5:]),
    }

def load_intelli_skeleton(path: Path) -> np.ndarray:
    array = np.loadtxt(path, delimiter=",")
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 1:
        array = array[None, :]
    if array.shape[1] != 25 * 3:
        raise ValueError(
            f"{path.name}: expected 75 coordinates per frame, received {array.shape[1]}."
        )
    return array.reshape(len(array), 25, 3)

def map_intelli_to_coco(sequence_25: np.ndarray) -> np.ndarray:
    sequence_25 = np.asarray(sequence_25, dtype=np.float32)
    output = np.zeros((sequence_25.shape[0], 17, 3), dtype=np.float32)
    for kinect_index, coco_index in INTELLI_TO_COCO.items():
        output[:, coco_index, :] = sequence_25[:, kinect_index, :]
    return output

def build_intellirehab_dataset() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if not INTELLI_SIMPLIFIED_DIR.exists():
        raise FileNotFoundError(
            f"Missing IntelliRehabDS Simplified folder: {INTELLI_SIMPLIFIED_DIR}"
        )

    files = sorted(INTELLI_SIMPLIFIED_DIR.glob("*.txt"))
    if not files:
        raise FileNotFoundError(
            f"No IntelliRehabDS .txt files found in {INTELLI_SIMPLIFIED_DIR}"
        )

    X_rows, y_rows = [], []
    subject_rows, task_rows, sample_rows = [], [], []
    reasons = Counter()

    for path in tqdm(files, desc="IntelliRehabDS: building common angles"):
        try:
            metadata = parse_intelli_filename(path)
            correctness = metadata["correctness"]
            if correctness not in (1, 2):
                reasons["unexpected_label"] += 1
                continue
            label = 1 if correctness == 1 else 0

            skeleton = load_intelli_skeleton(path)
            coco = map_intelli_to_coco(skeleton)
            coco = normalize_skeleton_length(coco, TARGET_LEN)
            angles = compute_common_angle_timeseries(coco)
        except Exception as exc:
            reasons[type(exc).__name__] += 1
            continue

        X_rows.append(angles)
        y_rows.append(label)
        subject_rows.append(str(metadata["subject"]))
        task_rows.append(str(metadata["gesture"]))
        sample_rows.append(path.name)

    if not X_rows:
        raise RuntimeError(f"IntelliRehabDS produced no valid trials. Reasons: {reasons}")

    print("IntelliRehabDS exclusions:", reasons.most_common())
    return (
        np.stack(X_rows).astype(np.float32),
        np.asarray(y_rows, dtype=np.int8),
        np.asarray(subject_rows, dtype=str),
        np.asarray(task_rows, dtype=str),
        np.asarray(sample_rows, dtype=str),
    )


In [ ]:
# Dataset validation, caching, and fingerprinting
def array_fingerprint(
    X: np.ndarray,
    y: np.ndarray,
    subject_id: np.ndarray,
    task_id: np.ndarray,
    sample_id: np.ndarray,
) -> str:
    hasher = hashlib.sha256()
    X_contiguous = np.ascontiguousarray(X)
    y_contiguous = np.ascontiguousarray(y)
    hasher.update(str(X_contiguous.shape).encode())
    hasher.update(str(X_contiguous.dtype).encode())
    hasher.update(X_contiguous.view(np.uint8))
    hasher.update(y_contiguous.view(np.uint8))
    for array in (subject_id, task_id, sample_id):
        hasher.update("\n".join(map(str, array.tolist())).encode("utf-8"))
    return hasher.hexdigest()

def make_bundle(
    name: str,
    arrays: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray],
) -> DatasetBundle:
    X, y, subject_id, task_id, sample_id = arrays

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int8).reshape(-1)
    subject_id = np.asarray(subject_id).astype(str).reshape(-1)
    task_id = np.asarray(task_id).astype(str).reshape(-1)
    sample_id = np.asarray(sample_id).astype(str).reshape(-1)

    if X.ndim != 3 or X.shape[1:] != (100, 10):
        raise ValueError(f"{name}: expected X shape (N,100,10), received {X.shape}.")
    if not (len(X) == len(y) == len(subject_id) == len(task_id) == len(sample_id)):
        raise ValueError(f"{name}: arrays have inconsistent lengths.")
    if set(np.unique(y)).difference({0, 1}):
        raise ValueError(f"{name}: labels must be encoded as 0/1.")
    if not np.isfinite(X).all():
        raise ValueError(f"{name}: angle tensor contains non-finite values.")
    if len(np.unique(sample_id)) != len(sample_id):
        duplicates = (
            pd.Series(sample_id)[pd.Series(sample_id).duplicated()]
            .head()
            .tolist()
        )
        raise ValueError(f"{name}: duplicate sample identifiers found: {duplicates}")

    fingerprint = array_fingerprint(X, y, subject_id, task_id, sample_id)
    return DatasetBundle(
        name, X, y, subject_id, task_id, sample_id, fingerprint
    )

def save_bundle_cache(bundle: DatasetBundle, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        representation_version=np.asarray(REPRESENTATION_VERSION),
        X=bundle.X,
        y=bundle.y,
        subject_id=bundle.subject_id,
        task_id=bundle.task_id,
        sample_id=bundle.sample_id,
    )

def load_bundle_cache(name: str, path: Path) -> DatasetBundle:
    with np.load(path, allow_pickle=False) as archive:
        cached_version = str(archive["representation_version"].item())
        if cached_version != REPRESENTATION_VERSION:
            raise ValueError(
                f"{name}: cache version is {cached_version}, "
                f"expected {REPRESENTATION_VERSION}."
            )
        arrays = (
            archive["X"],
            archive["y"],
            archive["subject_id"],
            archive["task_id"],
            archive["sample_id"],
        )
    return make_bundle(name, arrays)

def load_verified_rehab() -> DatasetBundle:
    if not PRIMARY_REHAB_EXPORT.exists():
        raise FileNotFoundError(
            f"Missing verified REHAB24-6 export: {PRIMARY_REHAB_EXPORT}\n"
            "Generate it with the primary REHAB24-6 analysis notebook first."
        )

    with np.load(PRIMARY_REHAB_EXPORT, allow_pickle=False) as archive:
        required = {"X", "y", "subject_id", "task_id", "sample_id"}
        missing = required.difference(archive.files)
        if missing:
            raise KeyError(
                f"{PRIMARY_REHAB_EXPORT.name} is missing arrays: "
                f"{sorted(missing)}"
            )
        arrays = (
            np.asarray(archive["X"], dtype=np.float32),
            np.asarray(archive["y"], dtype=np.int8),
            np.asarray(archive["subject_id"]).astype(str),
            np.asarray(archive["task_id"]).astype(str),
            np.asarray(archive["sample_id"]).astype(str),
        )

    bundle = make_bundle("REHAB24-6", arrays)
    counts = dict(zip(*np.unique(bundle.y, return_counts=True)))

    assert bundle.X.shape == (1072, 100, 10)
    assert counts == {0: 504, 1: 568}
    assert len(np.unique(bundle.subject_id)) == 10
    assert len(np.unique(bundle.task_id)) == 6
    assert bundle.fingerprint.startswith(EXPECTED_REHAB_FINGERPRINT_PREFIX), (
        "REHAB24-6 fingerprint differs from the verified primary export. "
        f"Observed {bundle.fingerprint}."
    )

    print(
        "REHAB24-6 verified export:",
        f"X={bundle.X.shape}, labels={counts}, ",
        f"subjects={len(np.unique(bundle.subject_id))}, ",
        f"tasks={len(np.unique(bundle.task_id))}, ",
        f"fingerprint={bundle.fingerprint[:16]}...",
    )
    return bundle

def load_or_build_intelli() -> DatasetBundle:
    if INTELLI_CACHE.exists() and not REBUILD_INTELLI_PROCESSED_DATA:
        bundle = load_bundle_cache("IntelliRehabDS", INTELLI_CACHE)
        source = "validated cache"
    else:
        bundle = make_bundle("IntelliRehabDS", build_intellirehab_dataset())
        save_bundle_cache(bundle, INTELLI_CACHE)
        source = "raw Simplified files"

    counts = dict(zip(*np.unique(bundle.y, return_counts=True)))
    assert bundle.X.shape == (2577, 100, 10)
    assert counts == {0: 530, 1: 2047}
    assert len(np.unique(bundle.subject_id)) == 30
    assert len(np.unique(bundle.task_id)) == 9
    assert bundle.fingerprint == EXPECTED_INTELLI_FINGERPRINT, (
        "IntelliRehabDS fingerprint differs from the final manuscript analysis. "
        f"Observed {bundle.fingerprint}."
    )

    print(
        f"IntelliRehabDS loaded from {source}: ",
        f"X={bundle.X.shape}, labels={counts}, ",
        f"subjects={len(np.unique(bundle.subject_id))}, ",
        f"tasks={len(np.unique(bundle.task_id))}, ",
        f"fingerprint={bundle.fingerprint[:16]}...",
    )
    return bundle

datasets = {
    "REHAB24-6": load_verified_rehab(),
    "IntelliRehabDS": load_or_build_intelli(),
}


## Evaluation manifests

The same subject-disjoint generalized folds and repeated individualized subject-task splits are reused across original and shuffled conditions.

In [ ]:
# Shared split manifests
MANIFEST_VERSION = 2

def individualized_test_fraction(y_pair: np.ndarray) -> float:
    """
    Reproduce the allocation rule used in the supplied IntelliRehabDS notebook:
    keep at least two minority examples in both train and test whenever feasible.
    """
    y_pair = np.asarray(y_pair, dtype=int)
    n0 = int((y_pair == 0).sum())
    n1 = int((y_pair == 1).sum())
    n_total = len(y_pair)

    if n_total < MIN_TOTAL_COUNT:
        raise ValueError(f"too_small_total={n_total}")
    if min(n0, n1) < MIN_CLASS_COUNT:
        raise ValueError(f"min_class={min(n0, n1)}")

    n_minority = min(n0, n1)
    n_test_minority = max(2, int(round(BASE_TEST_FRACTION * n_minority)))
    n_test_minority = min(n_test_minority, n_minority - 2)

    if n_test_minority < 2:
        raise ValueError("cannot_allocate_two_minority_samples_to_both_partitions")

    fraction = max(BASE_TEST_FRACTION, n_test_minority / n_minority)
    return float(min(fraction, 0.49))

def generate_manifest(bundle: DatasetBundle) -> dict[str, Any]:
    generalized = []
    group_kfold = GroupKFold(n_splits=N_GENERALIZED_FOLDS)

    for fold, (train_idx, test_idx) in enumerate(
        group_kfold.split(bundle.X, bundle.y, groups=bundle.subject_id),
        start=1,
    ):
        if len(np.unique(bundle.y[train_idx])) < 2:
            raise ValueError(f"{bundle.name}: generalized fold {fold} has one training class.")
        if len(np.unique(bundle.y[test_idx])) < 2:
            raise ValueError(f"{bundle.name}: generalized fold {fold} has one test class.")

        generalized.append({
            "fold": fold,
            "train_idx": train_idx.astype(int).tolist(),
            "test_idx": test_idx.astype(int).tolist(),
            "train_subjects": sorted(np.unique(bundle.subject_id[train_idx]).tolist()),
            "test_subjects": sorted(np.unique(bundle.subject_id[test_idx]).tolist()),
        })

    individualized = []
    exclusions = []

    pair_frame = pd.DataFrame({
        "index": np.arange(len(bundle.y)),
        "subject_id": bundle.subject_id,
        "task_id": bundle.task_id,
        "y": bundle.y,
    })

    for (subject_id, task_id), group in pair_frame.groupby(
        ["subject_id", "task_id"], sort=True
    ):
        indices = group["index"].to_numpy(dtype=int)
        y_pair = bundle.y[indices]

        try:
            test_fraction = individualized_test_fraction(y_pair)
        except ValueError as exc:
            exclusions.append({
                "subject_id": str(subject_id),
                "task_id": str(task_id),
                "n_total": int(len(indices)),
                "n0": int((y_pair == 0).sum()),
                "n1": int((y_pair == 1).sum()),
                "reason": str(exc),
            })
            continue

        pair_seed = stable_seed(
            BASE_SEED,
            bundle.name,
            subject_id,
            task_id,
            "individualized_splits",
        )
        splitter = StratifiedShuffleSplit(
            n_splits=N_INDIVIDUALIZED_REPEATS,
            test_size=test_fraction,
            random_state=pair_seed,
        )

        for repeat, (train_local, test_local) in enumerate(
            splitter.split(np.zeros(len(indices)), y_pair),
            start=1,
        ):
            train_idx = indices[train_local]
            test_idx = indices[test_local]

            if len(np.unique(bundle.y[train_idx])) != 2:
                raise RuntimeError("Individualized training split lost a class.")
            if len(np.unique(bundle.y[test_idx])) != 2:
                raise RuntimeError("Individualized test split lost a class.")

            individualized.append({
                "subject_id": str(subject_id),
                "task_id": str(task_id),
                "repeat": repeat,
                "test_fraction": test_fraction,
                "train_idx": train_idx.astype(int).tolist(),
                "test_idx": test_idx.astype(int).tolist(),
            })

    return {
        "manifest_version": MANIFEST_VERSION,
        "dataset": bundle.name,
        "data_fingerprint": bundle.fingerprint,
        "split_configuration": {
            "n_generalized_folds": N_GENERALIZED_FOLDS,
            "n_individualized_repeats": N_INDIVIDUALIZED_REPEATS,
            "base_test_fraction": BASE_TEST_FRACTION,
            "min_class_count": MIN_CLASS_COUNT,
            "min_total_count": MIN_TOTAL_COUNT,
            "base_seed": BASE_SEED,
        },
        "generalized": generalized,
        "individualized": individualized,
        "individualized_exclusions": exclusions,
    }

def validate_manifest(bundle: DatasetBundle, manifest: dict[str, Any]) -> None:
    if manifest.get("manifest_version") != MANIFEST_VERSION:
        raise ValueError("Manifest version mismatch.")
    if manifest.get("dataset") != bundle.name:
        raise ValueError("Manifest dataset mismatch.")
    if manifest.get("data_fingerprint") != bundle.fingerprint:
        raise ValueError("Manifest fingerprint does not match the processed tensor.")

    n = len(bundle.y)

    for split in manifest["generalized"]:
        train_idx = np.asarray(split["train_idx"], dtype=int)
        test_idx = np.asarray(split["test_idx"], dtype=int)
        if np.intersect1d(train_idx, test_idx).size:
            raise ValueError("Generalized train/test overlap.")
        if train_idx.min() < 0 or test_idx.min() < 0:
            raise IndexError("Negative sample index in manifest.")
        if train_idx.max() >= n or test_idx.max() >= n:
            raise IndexError("Out-of-range sample index in manifest.")
        if set(bundle.subject_id[train_idx]).intersection(bundle.subject_id[test_idx]):
            raise ValueError("Generalized split is not subject-disjoint.")

    for split in manifest["individualized"]:
        train_idx = np.asarray(split["train_idx"], dtype=int)
        test_idx = np.asarray(split["test_idx"], dtype=int)
        if np.intersect1d(train_idx, test_idx).size:
            raise ValueError("Individualized train/test overlap.")
        combined = np.r_[train_idx, test_idx]
        if set(bundle.subject_id[combined]) != {str(split["subject_id"])}:
            raise ValueError("Individualized split contains multiple subjects.")
        if set(bundle.task_id[combined]) != {str(split["task_id"])}:
            raise ValueError("Individualized split contains multiple tasks.")

def save_manifest(manifest: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

def load_or_build_manifest(bundle: DatasetBundle, path: Path) -> dict[str, Any]:
    manifest = None

    if path.exists() and not REBUILD_MANIFESTS:
        try:
            candidate = json.loads(path.read_text(encoding="utf-8"))
            validate_manifest(bundle, candidate)
            manifest = candidate
            source = "existing manifest"
        except Exception as exc:
            warnings.warn(f"{bundle.name}: rebuilding incompatible manifest: {exc}")

    if manifest is None:
        manifest = generate_manifest(bundle)
        validate_manifest(bundle, manifest)
        save_manifest(manifest, path)
        source = "generated manifest"

    pair_keys = {
        (row["subject_id"], row["task_id"])
        for row in manifest["individualized"]
    }
    expected = EXPECTED_INDIVIDUALIZED_PAIRS.get(bundle.name)

    print(
        f"{bundle.name}: {source}; generalized folds={len(manifest['generalized'])}, "
        f"individualized pairs={len(pair_keys)}, "
        f"individualized split rows={len(manifest['individualized'])}."
    )

    if expected is not None and len(pair_keys) != expected:
        message = (
            f"{bundle.name}: obtained {len(pair_keys)} eligible pairs, "
            f"whereas the manuscript currently reports {expected}. "
            "Review the exclusions printed below before updating the manuscript."
        )
        if STRICT_PAIR_COUNT_CHECK:
            raise ValueError(message)
        warnings.warn(message)

    exclusions = pd.DataFrame(manifest["individualized_exclusions"])
    if len(exclusions):
        display(exclusions["reason"].value_counts().rename("count").to_frame())

    return manifest

manifests = {
    name: load_or_build_manifest(bundle, MANIFEST_FILES[name])
    for name, bundle in datasets.items()
}

# Final-manuscript eligibility checks.
for dataset_name, expected_pairs in EXPECTED_INDIVIDUALIZED_PAIRS.items():
    observed_pairs = len({
        (row["subject_id"], row["task_id"])
        for row in manifests[dataset_name]["individualized"]
    })
    assert observed_pairs == expected_pairs


In [ ]:
# Deterministic temporal-order stress test
def shuffle_time_within_each_trial(
    X: np.ndarray,
    *,
    dataset_name: str,
    seed: int,
) -> np.ndarray:
    """
    Independently permute frames within every trial.

    One shuffled tensor is generated per dataset and reused by every model,
    regime, and split.
    """
    X = np.asarray(X)
    shuffled = np.empty_like(X)
    rng = np.random.default_rng(stable_seed(seed, dataset_name, "temporal_shuffle"))
    for trial_index in range(len(X)):
        permutation = rng.permutation(X.shape[1])
        shuffled[trial_index] = X[trial_index, permutation, :]
    return shuffled

condition_arrays = {
    name: {
        "original": bundle.X,
        "shuffled": shuffle_time_within_each_trial(
            bundle.X,
            dataset_name=name,
            seed=SHUFFLE_SEED,
        ),
    }
    for name, bundle in datasets.items()
}

print("Created one deterministic shuffled tensor per dataset.")


## Fixed cross-model configurations

The four model configurations are fixed across datasets, regimes, and shuffle conditions. For each matched original/shuffled comparison, the same split indices and model seed are used; only temporal order changes.


In [ ]:
# Model fitting and prediction
def _class_weight_dict(y_train: np.ndarray) -> dict[int, float]:
    y_train = np.asarray(y_train, dtype=int)
    classes, counts = np.unique(y_train, return_counts=True)
    if set(classes) != {0, 1}:
        raise ValueError("Training data must contain both classes.")
    n = len(y_train)
    return {int(c): float(n / (2.0 * count)) for c, count in zip(classes, counts)}

def _standardize_sequence_channels(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return (
        ((X_train - mean) / std).astype(np.float32),
        ((X_test - mean) / std).astype(np.float32),
    )

def build_lstm(input_shape: tuple[int, int], seed: int) -> keras.Model:
    set_global_seed(seed)
    keras.backend.clear_session()

    inputs = keras.Input(shape=input_shape)
    x = keras.layers.LSTM(
        LSTM_CONFIG.units,
        return_sequences=False,
        kernel_initializer=keras.initializers.GlorotUniform(seed=seed),
        recurrent_initializer=keras.initializers.Orthogonal(seed=seed + 1),
    )(inputs)
    x = keras.layers.Dropout(LSTM_CONFIG.dropout, seed=seed + 2)(x)
    outputs = keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_initializer=keras.initializers.GlorotUniform(seed=seed + 3),
    )(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(LSTM_CONFIG.learning_rate),
        loss="binary_crossentropy",
    )
    return model

def fit_predict(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    *,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return hard predictions and continuous scores."""
    y_train = np.asarray(y_train, dtype=int)

    if model_name == "RF":
        model = RandomForestClassifier(
            n_estimators=RF_CONFIG.n_estimators,
            max_depth=RF_CONFIG.max_depth,
            min_samples_leaf=RF_CONFIG.min_samples_leaf,
            class_weight=RF_CONFIG.class_weight,
            n_jobs=RF_CONFIG.n_jobs,
            random_state=seed,
        )
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        score = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]

    elif model_name == "SVM":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(
                kernel="rbf",
                C=SVM_CONFIG.C,
                gamma=SVM_CONFIG.gamma,
                class_weight=SVM_CONFIG.class_weight,
            )),
        ])
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        score = model.decision_function(X_test.reshape(len(X_test), -1))

    elif model_name == "XGBoost":
        n_negative = int((y_train == 0).sum())
        n_positive = int((y_train == 1).sum())
        scale_pos_weight = n_negative / max(n_positive, 1)

        model = XGBClassifier(
            n_estimators=XGB_CONFIG.n_estimators,
            max_depth=XGB_CONFIG.max_depth,
            learning_rate=XGB_CONFIG.learning_rate,
            subsample=XGB_CONFIG.subsample,
            colsample_bytree=XGB_CONFIG.colsample_bytree,
            reg_lambda=XGB_CONFIG.reg_lambda,
            min_child_weight=XGB_CONFIG.min_child_weight,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            n_jobs=XGB_CONFIG.n_jobs,
            random_state=seed,
            tree_method="hist",
        )
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        score = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]

    elif model_name == "LSTM":
        Xtr, Xte = _standardize_sequence_channels(X_train, X_test)
        model = build_lstm((Xtr.shape[1], Xtr.shape[2]), seed=seed)
        batch_size = min(LSTM_CONFIG.batch_size, len(Xtr))
        model.fit(
            Xtr,
            y_train,
            epochs=LSTM_CONFIG.epochs,
            batch_size=batch_size,
            class_weight=_class_weight_dict(y_train),
            verbose=0,
            shuffle=False,
        )
        score = model.predict(Xte, verbose=0).reshape(-1)
        keras.backend.clear_session()
        del model
        gc.collect()

    else:
        raise KeyError(f"Unknown model: {model_name}")

    if model_name == "SVM":
        prediction = (score >= 0.0).astype(int)
    else:
        prediction = (score >= 0.5).astype(int)

    return prediction, np.asarray(score, dtype=float)


## Execute the cross-method experiment

This is the long-running cell. Results are checkpointed after every split-model-condition unit, so interrupted runs can be resumed safely.

Expected total completed units:

- REHAB24-6: `2 × 4 × (5 + 46×10) = 3720`
- IntelliRehabDS: `2 × 4 × (5 + 16×10) = 1320`
- Total: `5040`


In [ ]:
# Experiment execution with checkpointing
RESULTS_PATH = OUTPUT_DIR / "cross_method_split_level_results_CANONICAL.csv"

RESULT_COLUMNS = [
    "config_hash",
    "dataset",
    "data_fingerprint",
    "regime",
    "condition",
    "model",
    "unit_id",
    "fold",
    "subject_id",
    "task_id",
    "repeat",
    "n_train",
    "n_test",
    "balanced_accuracy",
    "roc_auc",
]

def load_completed_keys(path: Path) -> set[tuple[str, ...]]:
    if not path.exists():
        return set()
    frame = pd.read_csv(
        path,
        dtype={"config_hash": str, "data_fingerprint": str, "subject_id": str, "task_id": str},
    )
    frame = frame[frame["config_hash"] == CONFIG_HASH]
    return set(
        zip(
            frame["dataset"],
            frame["data_fingerprint"],
            frame["regime"],
            frame["condition"],
            frame["model"],
            frame["unit_id"],
        )
    )

def append_result(row: dict[str, Any], path: Path) -> None:
    output = pd.DataFrame([row], columns=RESULT_COLUMNS)
    output.to_csv(path, mode="a", header=not path.exists(), index=False)

def evaluate_one_split(
    *,
    bundle: DatasetBundle,
    X_condition: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    dataset_name: str,
    regime: str,
    condition: str,
    model_name: str,
    unit_id: str,
    fold: int | None,
    subject_id: str | None,
    task_id: str | None,
    repeat: int | None,
) -> dict[str, Any]:
    y_train = bundle.y[train_idx]
    y_test = bundle.y[test_idx]

    # Use the same model/randomness seed for original and shuffled inputs so the
    # paired stress test changes temporal order, not model initialization.
    seed = stable_seed(
        BASE_SEED,
        dataset_name,
        regime,
        model_name,
        unit_id,
    )
    prediction, score = fit_predict(
        model_name,
        X_condition[train_idx],
        y_train,
        X_condition[test_idx],
        seed=seed,
    )

    ba = balanced_accuracy_score(y_test, prediction)
    auc = roc_auc_score(y_test, score) if len(np.unique(y_test)) == 2 else np.nan

    return {
        "config_hash": CONFIG_HASH,
        "dataset": dataset_name,
        "data_fingerprint": bundle.fingerprint,
        "regime": regime,
        "condition": condition,
        "model": model_name,
        "unit_id": unit_id,
        "fold": fold,
        "subject_id": subject_id,
        "task_id": task_id,
        "repeat": repeat,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "balanced_accuracy": float(ba),
        "roc_auc": float(auc),
    }

completed = load_completed_keys(RESULTS_PATH)
print(f"Resuming with {len(completed)} completed split-model-condition units.")

for dataset_name, bundle in datasets.items():
    manifest = manifests[dataset_name]

    for condition in CONDITIONS_TO_RUN:
        X_condition = condition_arrays[dataset_name][condition]

        if "generalized" in REGIMES_TO_RUN:
            for split in manifest["generalized"]:
                fold = int(split["fold"])
                train_idx = np.asarray(split["train_idx"], dtype=int)
                test_idx = np.asarray(split["test_idx"], dtype=int)
                unit_id = f"fold={fold}"

                for model_name in MODELS_TO_RUN:
                    key = (
                        dataset_name,
                        bundle.fingerprint,
                        "generalized",
                        condition,
                        model_name,
                        unit_id,
                    )
                    if key in completed:
                        continue

                    row = evaluate_one_split(
                        bundle=bundle,
                        X_condition=X_condition,
                        train_idx=train_idx,
                        test_idx=test_idx,
                        dataset_name=dataset_name,
                        regime="generalized",
                        condition=condition,
                        model_name=model_name,
                        unit_id=unit_id,
                        fold=fold,
                        subject_id=None,
                        task_id=None,
                        repeat=None,
                    )
                    append_result(row, RESULTS_PATH)
                    completed.add(key)
                    print(dataset_name, "generalized", condition, model_name, unit_id, row["balanced_accuracy"])

        if "individualized" in REGIMES_TO_RUN:
            for split in manifest["individualized"]:
                subject_id = str(split["subject_id"])
                task_id = str(split["task_id"])
                repeat = int(split["repeat"])
                train_idx = np.asarray(split["train_idx"], dtype=int)
                test_idx = np.asarray(split["test_idx"], dtype=int)
                unit_id = f"subject={subject_id}|task={task_id}|repeat={repeat}"

                for model_name in MODELS_TO_RUN:
                    key = (
                        dataset_name,
                        bundle.fingerprint,
                        "individualized",
                        condition,
                        model_name,
                        unit_id,
                    )
                    if key in completed:
                        continue

                    row = evaluate_one_split(
                        bundle=bundle,
                        X_condition=X_condition,
                        train_idx=train_idx,
                        test_idx=test_idx,
                        dataset_name=dataset_name,
                        regime="individualized",
                        condition=condition,
                        model_name=model_name,
                        unit_id=unit_id,
                        fold=None,
                        subject_id=subject_id,
                        task_id=task_id,
                        repeat=repeat,
                    )
                    append_result(row, RESULTS_PATH)
                    completed.add(key)
                    print(dataset_name, "individualized", condition, model_name, unit_id, row["balanced_accuracy"])

print("Finished. Split-level results:", RESULTS_PATH)


## Aggregate at the manuscript evaluation units

In [ ]:
# Aggregate at the correct evaluation unit
results = pd.read_csv(
    RESULTS_PATH,
    dtype={
        "config_hash": str,
        "data_fingerprint": str,
        "subject_id": str,
        "task_id": str,
    },
)
results = results[results["config_hash"] == CONFIG_HASH].copy()

# Retain only rows belonging to the exact dataset tensors loaded above.
current_fingerprints = {
    name: bundle.fingerprint
    for name, bundle in datasets.items()
}
results = results[
    results.apply(
        lambda row: (
            row["dataset"] in current_fingerprints
            and row["data_fingerprint"]
            == current_fingerprints[row["dataset"]]
        ),
        axis=1,
    )
].copy()

results = results.drop_duplicates(
    subset=[
        "dataset", "data_fingerprint", "regime",
        "condition", "model", "unit_id",
    ],
    keep="last",
).reset_index(drop=True)

expected_rows = {}
for dataset_name in datasets:
    manifest = manifests[dataset_name]
    expected_rows[dataset_name] = (
        len(CONDITIONS_TO_RUN)
        * len(MODELS_TO_RUN)
        * (
            len(manifest["generalized"])
            + len(manifest["individualized"])
        )
    )

for dataset_name, expected in expected_rows.items():
    observed = int((results["dataset"] == dataset_name).sum())
    if observed != expected:
        raise RuntimeError(
            f"{dataset_name}: incomplete canonical results "
            f"({observed}/{expected} rows). Finish the execution cell first."
        )

expected_models = set(MODELS_TO_RUN)
observed_models = set(results["model"].unique())
missing_models = expected_models.difference(observed_models)
if missing_models:
    warnings.warn(f"Results are incomplete; missing models: {sorted(missing_models)}")

generalized_units = results[results["regime"] == "generalized"].copy()

individualized_repeats = results[results["regime"] == "individualized"].copy()
individualized_pairs = (
    individualized_repeats
    .groupby(
        ["dataset", "condition", "model", "subject_id", "task_id"],
        as_index=False,
        dropna=False,
    )
    .agg(
        balanced_accuracy=("balanced_accuracy", "mean"),
        roc_auc=("roc_auc", "mean"),
        n_repeats=("repeat", "nunique"),
    )
)

PAIR_SUMMARY_PATH = OUTPUT_DIR / "cross_method_individualized_pair_summary.csv"
individualized_pairs.to_csv(PAIR_SUMMARY_PATH, index=False)

def summarize_units(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame
        .groupby(["dataset", "regime", "condition", "model"], as_index=False)
        .agg(
            mean_BA=("balanced_accuracy", "mean"),
            sd_BA=("balanced_accuracy", "std"),
            median_BA=("balanced_accuracy", "median"),
            n_units=("balanced_accuracy", "size"),
            mean_AUC=("roc_auc", "mean"),
            sd_AUC=("roc_auc", "std"),
        )
    )

generalized_summary = summarize_units(generalized_units)

individualized_for_summary = individualized_pairs.copy()
individualized_for_summary["regime"] = "individualized"
individualized_summary = summarize_units(individualized_for_summary)

summary = pd.concat([generalized_summary, individualized_summary], ignore_index=True)
SUMMARY_PATH = OUTPUT_DIR / "cross_method_manuscript_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False)

display(summary.sort_values(["dataset", "regime", "condition", "model"]))
print("Saved pair-level summary:", PAIR_SUMMARY_PATH)
print("Saved manuscript summary:", SUMMARY_PATH)


## Paired temporal-shuffle effects

In [ ]:
# Paired original-versus-shuffled effects at the correct unit
from scipy.stats import wilcoxon

def bootstrap_median_ci(values: np.ndarray, seed: int, n_boot: int = 5000) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot[i] = np.median(sample)
    return tuple(np.quantile(boot, [0.025, 0.975]))

effect_rows = []

# Generalized: paired by outer fold
for (dataset, model), group in generalized_units.groupby(["dataset", "model"]):
    wide = group.pivot(index="fold", columns="condition", values="balanced_accuracy").dropna()
    if {"original", "shuffled"}.issubset(wide.columns):
        delta = (wide["shuffled"] - wide["original"]).to_numpy()
        p_value = wilcoxon(delta).pvalue if np.any(delta != 0) else 1.0
        ci_low, ci_high = bootstrap_median_ci(
            delta,
            stable_seed(dataset, model, "generalized", "bootstrap"),
        )
        effect_rows.append({
            "dataset": dataset,
            "regime": "generalized",
            "model": model,
            "n_units": len(delta),
            "mean_delta_shuffled_minus_original": delta.mean(),
            "median_delta_shuffled_minus_original": np.median(delta),
            "median_ci_low": ci_low,
            "median_ci_high": ci_high,
            "wilcoxon_p": p_value,
        })

# Individualized: paired by subject-task pair after averaging repeats
for (dataset, model), group in individualized_pairs.groupby(["dataset", "model"]):
    wide = group.pivot(
        index=["subject_id", "task_id"],
        columns="condition",
        values="balanced_accuracy",
    ).dropna()
    if {"original", "shuffled"}.issubset(wide.columns):
        delta = (wide["shuffled"] - wide["original"]).to_numpy()
        p_value = wilcoxon(delta).pvalue if np.any(delta != 0) else 1.0
        ci_low, ci_high = bootstrap_median_ci(
            delta,
            stable_seed(dataset, model, "individualized", "bootstrap"),
        )
        effect_rows.append({
            "dataset": dataset,
            "regime": "individualized",
            "model": model,
            "n_units": len(delta),
            "mean_delta_shuffled_minus_original": delta.mean(),
            "median_delta_shuffled_minus_original": np.median(delta),
            "median_ci_low": ci_low,
            "median_ci_high": ci_high,
            "wilcoxon_p": p_value,
        })

shuffle_effects = pd.DataFrame(effect_rows)
SHUFFLE_EFFECTS_PATH = OUTPUT_DIR / "cross_method_shuffle_effects.csv"
shuffle_effects.to_csv(SHUFFLE_EFFECTS_PATH, index=False)

display(shuffle_effects.sort_values(["dataset", "regime", "model"]))
print("Saved shuffle effects:", SHUFFLE_EFFECTS_PATH)


## Subject-aware temporal-shuffle inference

In [ ]:
# Subject-aware individualized temporal-shuffle sensitivity
#
# Multiple subject-task pairs may originate from the same participant.
# We therefore average pair-level shuffle deltas within subject and perform
# an exact exhaustive two-sided sign-flip test over subjects. Holm adjustment
# is applied across the four model-family tests within each dataset.

def exact_two_sided_sign_flip_p(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return np.nan
    if np.allclose(values, 0.0):
        return 1.0

    observed = abs(values.mean())
    exceed = 0
    total = 1 << n

    for mask in range(total):
        signs = np.ones(n, dtype=float)
        for i in range(n):
            if mask & (1 << i):
                signs[i] = -1.0
        statistic = abs(np.mean(signs * values))
        if statistic >= observed - 1e-15:
            exceed += 1

    return exceed / total

def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, idx in enumerate(order):
        raw_adjusted = (m - rank) * p_values[idx]
        running_max = max(running_max, raw_adjusted)
        adjusted_sorted[rank] = min(1.0, running_max)

    adjusted = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        adjusted[idx] = adjusted_sorted[rank]
    return adjusted

subject_effect_rows = []

for (dataset, model), group in individualized_pairs.groupby(
    ["dataset", "model"]
):
    wide = group.pivot(
        index=["subject_id", "task_id"],
        columns="condition",
        values="balanced_accuracy",
    ).dropna()

    pair_delta = (
        wide["shuffled"] - wide["original"]
    ).rename("delta").reset_index()

    subject_delta = (
        pair_delta
        .groupby("subject_id", as_index=False)["delta"]
        .mean()
    )

    values = subject_delta["delta"].to_numpy(dtype=float)
    exact_p = exact_two_sided_sign_flip_p(values)

    subject_effect_rows.append({
        "dataset": dataset,
        "model": model,
        "n_subjects": len(values),
        "mean_subject_delta_shuffled_minus_original": values.mean(),
        "median_subject_delta_shuffled_minus_original": np.median(values),
        "exact_sign_flip_p": exact_p,
    })

subject_aware = pd.DataFrame(subject_effect_rows)

# Holm correction across the four model families separately within each dataset.
subject_aware["holm_p"] = np.nan
for dataset, idx in subject_aware.groupby("dataset").groups.items():
    idx = list(idx)
    subject_aware.loc[idx, "holm_p"] = holm_adjust(
        subject_aware.loc[idx, "exact_sign_flip_p"].to_numpy()
    )

SUBJECT_AWARE_PATH = (
    OUTPUT_DIR / "cross_method_subject_aware_shuffle_sensitivity.csv"
)
subject_aware.to_csv(SUBJECT_AWARE_PATH, index=False)

display(
    subject_aware.sort_values(["dataset", "model"]).reset_index(drop=True)
)
print("Saved subject-aware shuffle sensitivity:", SUBJECT_AWARE_PATH)


## Manuscript-ready tables

In [ ]:
# Manuscript-ready cross-method performance table (Table 20)
def formatted_value(mean: float, sd: float) -> str:
    return rf"${mean:.3f} \pm {sd:.3f}$"

table_source = summary.copy()
table_source["formatted"] = [
    formatted_value(mean, sd)
    for mean, sd in zip(table_source["mean_BA"], table_source["sd_BA"])
]

for dataset_name in ["REHAB24-6", "IntelliRehabDS"]:
    subset = table_source[table_source["dataset"] == dataset_name]
    table = subset.pivot(
        index="model",
        columns=["regime", "condition"],
        values="formatted",
    )
    desired_columns = [
        ("generalized", "original"),
        ("generalized", "shuffled"),
        ("individualized", "original"),
        ("individualized", "shuffled"),
    ]
    table = table.reindex(
        index=MODELS_TO_RUN,
        columns=desired_columns,
    )
    display(table)

    safe_name = dataset_name.lower().replace("-", "_")
    latex_path = OUTPUT_DIR / f"{safe_name}_cross_method_table.tex"
    latex_path.write_text(
        table.to_latex(
            escape=False,
            column_format="lcccc",
            multicolumn=True,
            multicolumn_format="c",
            na_rep="--",
        ),
        encoding="utf-8",
    )
    print("Saved:", latex_path)

# Compact subject-aware table (Table 21)
subject_table = subject_aware.copy()
subject_table["mean_delta"] = subject_table[
    "mean_subject_delta_shuffled_minus_original"
].map(lambda x: f"{x:+.3f}")
subject_table["exact_p"] = subject_table[
    "exact_sign_flip_p"
].map(lambda x: f"{x:.4f}")
subject_table["holm_p_fmt"] = subject_table[
    "holm_p"
].map(lambda x: f"{x:.4f}")

subject_table = subject_table[
    ["dataset", "model", "n_subjects", "mean_delta", "exact_p", "holm_p_fmt"]
].rename(columns={
    "n_subjects": "n",
    "mean_delta": "Mean subject ΔBA",
    "exact_p": "Exact p",
    "holm_p_fmt": "Holm p",
})

display(subject_table)

SUBJECT_TABLE_TEX = OUTPUT_DIR / "cross_method_subject_aware_shuffle_table.tex"
SUBJECT_TABLE_TEX.write_text(
    subject_table.to_latex(index=False, escape=False),
    encoding="utf-8",
)
print("Saved:", SUBJECT_TABLE_TEX)


## Expected manuscript-level values

A complete run should reproduce the final cross-method summary approximately as follows:

| Dataset | Model | Generalized original | Generalized shuffled | Individualized original | Individualized shuffled |
|---|---:|---:|---:|---:|---:|
| REHAB24-6 | RF | .513±.037 | .508±.028 | .878±.082 | .777±.158 |
| REHAB24-6 | SVM | .563±.030 | .516±.016 | .861±.121 | .731±.190 |
| REHAB24-6 | XGBoost | .520±.025 | .500±.035 | .709±.138 | .612±.125 |
| REHAB24-6 | LSTM | .546±.043 | .553±.019 | .874±.104 | .883±.095 |
| IntelliRehabDS | RF | .630±.071 | .590±.054 | .788±.197 | .753±.202 |
| IntelliRehabDS | SVM | .731±.037 | .624±.028 | .802±.185 | .729±.211 |
| IntelliRehabDS | XGBoost | .716±.048 | .671±.067 | .626±.173 | .616±.160 |
| IntelliRehabDS | LSTM | .672±.054 | .607±.100 | .801±.183 | .823±.170 |

The subject-aware individualized shuffle analysis should reproduce:

- **REHAB24-6 (9 subjects):** RF ΔBA ≈ −.099, exact p=.0039, Holm p=.0156; SVM ≈ −.132, .0039, .0156; XGBoost ≈ −.099, .0039, .0156; LSTM ≈ +.008, .6289, .6289.
- **IntelliRehabDS (7 subjects):** RF ≈ −.043, .1094, .3281; SVM ≈ −.094, .0156, .0625; XGBoost ≈ −.008, .5625, .6250; LSTM ≈ +.033, .3125, .6250.

Small floating-point differences may occur across TensorFlow/hardware versions, especially for the LSTM.


In [ ]:
# Reproducibility report
report = {
    "configuration_hash": CONFIG_HASH,
    "configuration": CONFIG_PAYLOAD,
    "angle_names": ANGLE_NAMES,
    "data_sources": {
        "REHAB24-6": {
            "verified_primary_export": str(PRIMARY_REHAB_EXPORT),
            "data_fingerprint": datasets["REHAB24-6"].fingerprint,
            "n_trials": int(len(datasets["REHAB24-6"].y)),
            "n_subjects": int(len(np.unique(datasets["REHAB24-6"].subject_id))),
            "n_tasks": int(len(np.unique(datasets["REHAB24-6"].task_id))),
        },
        "IntelliRehabDS": {
            "simplified_directory": str(INTELLI_SIMPLIFIED_DIR),
            "processed_cache": str(INTELLI_CACHE),
            "data_fingerprint": datasets["IntelliRehabDS"].fingerprint,
            "n_trials": int(len(datasets["IntelliRehabDS"].y)),
            "n_subjects": int(len(np.unique(datasets["IntelliRehabDS"].subject_id))),
            "n_tasks": int(len(np.unique(datasets["IntelliRehabDS"].task_id))),
        },
    },
    "manifests": {
        name: str(MANIFEST_FILES[name])
        for name in datasets
    },
    "eligible_individualized_pairs": {
        name: len({
            (row["subject_id"], row["task_id"])
            for row in manifests[name]["individualized"]
        })
        for name in datasets
    },
    "outputs": {
        "split_level_results": str(RESULTS_PATH),
        "individualized_pair_summary": str(PAIR_SUMMARY_PATH),
        "manuscript_summary": str(SUMMARY_PATH),
        "pair_level_shuffle_effects": str(SHUFFLE_EFFECTS_PATH),
        "subject_aware_shuffle_effects": str(SUBJECT_AWARE_PATH),
    },
}

REPORT_PATH = OUTPUT_DIR / "cross_method_reproducibility_report.json"
REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)
print("Saved:", REPORT_PATH)


## Interpretation

The shuffle experiment is a **predictive temporal-order stress test**. A decrease after shuffling indicates that the model's discrimination depends on the ordered phase alignment present in the normalized trajectories. It is not, by itself, a direct measure of learned biomechanical temporal reasoning.

For individualized analysis, the subject-aware sign-flip test is the inferential result used to account for multiple subject-task pairs originating from the same participant.
